In [1]:
%pwd

'C:\\Users\\User\\Documents\\Database Project'

In [2]:
!pip install pymysql sqlalchemy pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

DB_USER = "root"
DB_PASS = "1234"
DB_HOST = "127.0.0.1"
DB_PORT = "3306"
DB_NAME = "librarymS"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DATABASE();")).fetchone()[0])


Connected to: libraryms


In [4]:
print("BP1: Add New Member with Borrowing (no book yet)")

try:
    add_member = text("""
    INSERT INTO Members (member_id, name, email, phone, address)
    VALUES (:mid, :name, :email, :phone, :addr);
    """)

    add_borrowing = text("""
    INSERT INTO Borrowings (borrow_id, member_id, isbn, borrow_date, due_date, return_date)
    VALUES (:bid, :mid, NULL, :bdate, :ddate, NULL);
    """)

    params_member = {
        "mid": 100,
        "name": "Jidan",
        "email": "jidan100@gmail.com",
        "phone": "+44 1234567890",
        "addr": "123 Test Street"
    }
    params_borrow = {
        "bid": 100,
        "mid": 100,
        "bdate": "2025-04-01",
        "ddate": "2025-04-15"
    }

    with engine.begin() as conn:
        conn.execute(add_member, params_member)
        conn.execute(add_borrowing, params_borrow)

    print("Inserted BP1 successfully")

    print(pd.read_sql(text("SELECT * FROM Members WHERE member_id=100;"), engine))
    print(pd.read_sql(text("SELECT * FROM Borrowings WHERE borrow_id=100;"), engine))

except Exception as e:
    print("BP1 Error:", e)


=== BP1: Add New Member with Borrowing (no book yet) ===
Inserted BP1 successfully
   member_id   name               email           phone          address
0        100  Jidan  jidan100@gmail.com  +44 1234567890  123 Test Street
   borrow_id  member_id  isbn borrow_date    due_date return_date
0        100        100  None  2025-04-01  2025-04-15        None


In [5]:
print("BP2A: Total BOOKS borrowed per member (excludes pending NULL ISBN)")

bp2A = text("""
SELECT 
    m.member_id,
    m.name AS member_name,
    COUNT(b.borrow_id) AS total_books_borrowed
FROM Members m
LEFT JOIN Borrowings b 
    ON m.member_id = b.member_id
   AND b.borrow_date BETWEEN '2025-01-01' AND '2025-03-31'
   AND b.isbn IS NOT NULL
GROUP BY m.member_id, m.name
ORDER BY total_books_borrowed DESC;
""")

print(pd.read_sql(bp2A, engine).head(10))


BP2A: Total BOOKS borrowed per member (excludes pending NULL ISBN)
   member_id            member_name  total_books_borrowed
0         25          Sandra Storey                     6
1         24           Damian Mills                     5
2         15            Scott White                     4
3          1         Julian Collier                     3
4          6       Mrs Rosie Clarke                     3
5         11         Patricia Jones                     3
6         17       Miss Mary Fuller                     3
7         22  Sally Noble-Dickinson                     3
8          2          Dorothy Dixon                     2
9         19           Fiona Wilson                     2


In [12]:
print("BP3: Borrowed books for Member ID 25")

bp3 = text("""
SELECT 
    b.title AS book_title,
    br.borrow_date,
    br.due_date,
    br.return_date,
    CASE 
        WHEN br.return_date IS NULL AND br.due_date < CURDATE() THEN 'Overdue'
        WHEN br.return_date > br.due_date THEN 'Returned Late'
        ELSE 'On Time'
    END AS status
FROM Borrowings br
JOIN Books b ON br.isbn = b.isbn
WHERE br.member_id = 25
ORDER BY br.borrow_date;
""")

print(pd.read_sql(bp3, engine))


BP3: Borrowed books for Member ID 25
               book_title borrow_date    due_date return_date         status
0       Nostrum explicabo  2024-12-30  2025-01-13        None        Overdue
1         Adipisci veniam  2025-01-03  2025-01-17  2025-01-23  Returned Late
2            Facere magni  2025-01-07  2025-01-21  2025-01-27  Returned Late
3       Tempora in soluta  2025-01-26  2025-02-09        None        Overdue
4  Occaecati occaecati et  2025-02-03  2025-02-17  2025-02-28  Returned Late
5     Officia commodi nam  2025-02-04  2025-02-18  2025-02-26  Returned Late
6  Occaecati occaecati et  2025-03-01  2025-03-15  2025-03-19  Returned Late


In [13]:
print("BP4: Books in Technology genre")

bp4 = text("""
SELECT 
    b.title AS book_title,
    b.isbn,
    b.copies_available,
    g.genre_name,
    GROUP_CONCAT(DISTINCT a.name ORDER BY a.name SEPARATOR ', ') AS authors
FROM Books b
JOIN BookGenres bg ON b.isbn = bg.isbn
JOIN Genres g ON bg.genre_id = g.genre_id
LEFT JOIN BookAuthors ba ON b.isbn = ba.isbn
LEFT JOIN Authors a ON ba.author_id = a.author_id
WHERE g.genre_name = 'Technology'
GROUP BY b.isbn, b.title, b.copies_available, g.genre_name
ORDER BY b.title;
""")

print(pd.read_sql(bp4, engine))


BP4: Books in Technology genre
          book_title               isbn  copies_available  genre_name  \
0    Adipisci veniam  978-1-238-23771-0                10  Technology   
1  Beatae voluptates  978-1-967139-34-7                10  Technology   
2          Cum sequi  978-0-15-097431-4                 4  Technology   
3    Delectus itaque  978-1-4903-6824-5                 5  Technology   
4       Facere magni  978-1-72950-443-7                 7  Technology   
5  Tempora in soluta  978-0-914811-43-5                 4  Technology   

                        authors  
0                   Dr Liam Ali  
1                   Dr Liam Ali  
2            Dr Tracy Stevenson  
3            Dr Tracy Stevenson  
4                Toby Carpenter  
5  Deborah Adams, Mr Shane Hart  


In [14]:
print("BP5: Remove from circulation + list unavailable")

try:
    # show current
    print(pd.read_sql(text("""
        SELECT isbn, title, status, copies_available 
        FROM Books WHERE isbn='978-1-71363-639-7';
    """), engine))

    # update to repair
    with engine.begin() as conn:
        conn.execute(text("""
            UPDATE Books
            SET status='Repair', copies_available=0
            WHERE isbn='978-1-71363-639-7';
        """))

    # list unavailable
    unavailable = text("""
    SELECT isbn, title, status, copies_available
    FROM Books
    WHERE status <> 'Available' OR copies_available = 0
    ORDER BY status, title;
    """)

    print(pd.read_sql(unavailable, engine))

except Exception as e:
    print("BP5 Error:", e)


BP5: Remove from circulation + list unavailable
                isbn                     title     status  copies_available
0  978-1-71363-639-7  Similique similique quia  Available                 1
                isbn                     title  status  copies_available
0  978-1-71363-639-7  Similique similique quia  Repair                 0
